# 실습 3. OpenAI API 활용 프로그램 (Day 2 / 120분)

> 시나리오: **리뷰 감성 분류를 LLM 으로 다시 풀고, sklearn(실습 2)과 비교**
>
> 이 노트북은 외부 예제 없이 `openai` 패키지만으로 진행합니다.

## Task
1. 단발 호출 / 토큰·비용 / 스트리밍 (`.env` 의 `OPENAI_API_KEY`)
2. 리뷰 100개 긍/부정 분류 — `temperature=0`, JSON 응답 강제 → **실습 2 와 비교**
3. 비용 측정 (`response.usage`) → **1만 건 가정 시 비용 추산**

## 0) 준비 — `.env` 로드 + 클라이언트

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                 # .env 의 OPENAI_API_KEY 를 환경변수로 로드
client = OpenAI()             # 키는 환경변수에서 자동으로 읽음
MODEL = "gpt-4o-mini"
print("client ready:", MODEL)

client ready: gpt-4o-mini


## 1) Task 1 — 단발 호출 + 토큰/비용 확인

In [24]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "당신은 간결하게 답하는 도우미입니다."},
        {"role": "user", "content": "사내 챗봇을 한 문장으로 소개해줘."},
    ],
    temperature=1.0,
)
print(resp.choices[0].message.content)
print("usage:", resp.usage)    # prompt_tokens / completion_tokens / total_tokens

사내 챗봇은 직원의 질문에 신속하게 답변하고 업무 효율성을 높여주는 인공지능 기반의 대화형 도구입니다.
usage: CompletionUsage(completion_tokens=36, prompt_tokens=39, total_tokens=75, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


### 스트리밍 — 토큰이 생성되는 대로 출력

In [28]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "LLM 을 3줄로 설명해줘."}],
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    print(delta, end="")
print()

LLM(대형 언어 모델)은 방대한 데이터를 기반으로 자연어를 이해하고 생성하는 인공지능 모델입니다.


## 2) Task 2 — 리뷰 100개 긍/부정 분류 (LLM)

실습 1의 산출물 `reviews_clean.parquet` 에서 100건을 샘플링해 분류한다.
- `temperature=0` (분류는 결정적이어야 함)
- **JSON 응답 강제** (`response_format`) — 파싱 안전

In [29]:
import pandas as pd

df = pd.read_parquet("../data/reviews_clean.parquet")
df = df.dropna(subset=["rating"]).copy()
df = df[df["rating"] != 3]                      # 중립 제외 (실습 2 와 동일 기준)
df["label"] = (df["rating"] >= 4).astype(int)   # 정답: 4~5 긍정(1), 1~2 부정(0)
# 샘플의 원본 인덱스를 보관해 train/validation 분할에 사용 (reset_index 는 UI용)
sample = df.sample(100, random_state=42)
sample_idx = sample.index
sample = sample.reset_index(drop=True)
print(sample["label"].value_counts())
sample[["clean_text", "label"]].head()

label
1    60
0    40
Name: count, dtype: int64


,clean_text,label
0,재구매 의사 100 강추합니다,1
1,사이즈도 딱 맞고 마감이 깔끔해요,1
2,가성비 최고입니다 또 살게요,1
3,색감이 사진이랑 똑같아서 만족합니다,1
4,배송이 일주일이나 걸렸습니다 별로,0


In [42]:
import json

SYSTEM = (
    "너는 한국어 쇼핑몰 리뷰 감성 분류기다. "
    "입력 리뷰가 긍정이면 1, 부정이면 0 을 고른다. "
    'JSON 으로만 답한다: {"label": 0 또는 1}'
)

def classify(text: str) -> tuple[int, int]:   # (라벨, 사용 토큰) 을 함께 반환
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},   # JSON 강제
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": text},
        ],
    )
    data = json.loads(resp.choices[0].message.content)
    return int(data["label"]), resp.usage.total_tokens   # 라벨 + 비용계산용 토큰

# 1건 점검 — (라벨, 토큰) 튜플이 찍힌다
print(sample.loc[0, "clean_text"])
label, used = classify(sample.loc[0, "clean_text"])
print("라벨:", label, "/ 토큰:", used)

재구매 의사 100 강추합니다
라벨: 1 / 토큰: 79


In [47]:
import time

preds, tokens = [], 0
start = time.perf_counter()
for t in sample["clean_text"]:
    label, used = classify(t)
    preds.append(label)
    tokens += used
end = time.perf_counter()
llm_time = end - start
sample["pred"] = preds
print("총 토큰:", tokens)

총 토큰: 8130


### sklearn(실습 2) 과 비교 — 정확도

In [45]:
from sklearn.metrics import accuracy_score, classification_report

acc = accuracy_score(sample["label"], sample["pred"])
print(f"LLM 정확도: {acc:.3f}")
print(classification_report(sample["label"], sample["pred"], digits=3))
# TODO: 실습 2 sklearn 정확도와 한 표로 비교 (정확도 / 비용 / 속도 / 일관성)

LLM 정확도: 1.000
              precision    recall  f1-score   support

           0      1.000     1.000     1.000        40
           1      1.000     1.000     1.000        60

    accuracy                          1.000       100
   macro avg      1.000     1.000     1.000       100
weighted avg      1.000     1.000     1.000       100



## 3) Task 3 — 비용 측정 + 1만 건 추산

In [48]:
# gpt-4o-mini 단가 (2026 기준, 변동 가능) — 슬라이드 '비용·운영 한눈에' 참고
# 입력 0.15/M, 출력 0.60/M $. 분류는 출력이 5토큰 내외로 매우 짧아 출력 비용은 무시하고
# total_tokens 를 입력 단가로 보수적으로 추정한다.
PRICE_IN = 0.15 / 1_000_000   # 입력 토큰당 $

avg_tokens = tokens / len(sample)
cost_100   = tokens * PRICE_IN                 # 보수적으로 입력 단가 적용
print(f"평균 토큰/건: {avg_tokens:.1f}")
print(f"100건 비용: ${cost_100:.4f}")
print(f"1만 건 추산: ${cost_100 * 100:.2f}")

# 한 줄 결론
print("한 줄 결론: 대량·단순 분류는 비용·속도 측면에서 sklearn 같은 ML 모델이 유리하고, 적은 데이터나 복잡한 문맥·유연한 규칙이 필요할 때는 LLM을 선택하라.")

평균 토큰/건: 81.3
100건 비용: $0.0012
1만 건 추산: $0.12
한 줄 결론: 대량·단순 분류는 비용·속도 측면에서 sklearn 같은 ML 모델이 유리하고, 적은 데이터나 복잡한 문맥·유연한 규칙이 필요할 때는 LLM을 선택하라.


## 회고 / 산출물
- [ ] 비교표: ML vs LLM (정확도 / 비용 / 속도 / 일관성)
- [ ] 한 줄 결론 — *대량·단순 분류는 ___, 유연·복잡 추론은 ___*

In [50]:
# 세 문장(반어 리뷰)을 LLM과 간단한 ML 모델로 분류하고, 신뢰도와 비교표를 출력합니다\n
반어_리뷰 = [
    "와 정말 좋네요 한 번 쓰고 바로 고장나서 아주 만족합니다",
    "품질 최고예요~ 환불하고 싶을 만큼",
    "빠른 배송 감사합니다 일주일이나 걸려서요",
]

import json
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score
import numpy as np

def llm_classify_with_confidence(text: str) -> tuple:
    SYSTEM2 = (
        "너는 한국어 쇼핑몰 리뷰 감성 분류기다. "
        "입력 리뷰가 긍정이면 1, 부정이면 0 을 고른다. "
        '"JSON 으로만 답한다: {\"label\": 0 또는 1, \"confidence\": 0.0-1.0}'
    )
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            temperature=0,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": SYSTEM2},
                {"role": "user", "content": text},
            ],
        )
        data = json.loads(resp.choices[0].message.content)
        label = int(data.get("label"))
        conf = float(data.get("confidence"))
        used = None
        try:
            used = resp.usage.total_tokens if hasattr(resp, 'usage') else resp['usage'].get('total_tokens')
        except Exception:
            used = None
        return label, conf, used
    except Exception as e:
        return None, None, None

# LLM로 3개 리뷰 분류
llm_results = []
total_tokens_used = 0
for r in 반어_리뷰:
    lbl, conf, used = llm_classify_with_confidence(r)
    if used:
        try:
            total_tokens_used += int(used)
        except Exception:
            pass
    llm_results.append({"review": r, "label": lbl, "confidence": conf, "tokens": used})

for res in llm_results:
    print("LLM:", res["review"])
    if res["label"] is not None:
        print(f'  라벨: {res["label"]}, 신뢰도: {res["confidence"]:.3f}, 토큰: {res["tokens"]}')
    else:
        print('  분류 실패 (응답 오류)')
    print()

# ML: TF-IDF + LogisticRegression 간단 학습 (가능하면 기존 df, sample_idx 사용)
ml_results = []
if 'df' in globals():
    try:
        if 'sample_idx' in globals():
            train_df = df.drop(index=sample_idx)
        else:
            train_df = df.sample(frac=0.8, random_state=42)
        X_train = train_df['clean_text'].fillna('')
        y_train = train_df['label']
        model = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
        model.fit(X_train, y_train)
        probs = model.predict_proba(반어_리뷰)
        for p, r in zip(probs, 반어_리뷰):
            prob_pos = float(p[1])
            label = int(prob_pos >= 0.5)
            ml_results.append({"review": r, "label": label, "confidence": prob_pos})
        for res in ml_results:
            print("ML:", res["review"])
            print(f'  라벨: {res["label"]}, 신뢰도: {res["confidence"]:.3f}')
            print()
    except Exception as e:
        print('ML 모델 학습/예측 중 오류:', e)
else:
    print('ML 학습에 필요한 `df`가 없습니다. 먼저 데이터 로드/전처리를 실행하세요.')

# 비교표 생성 (가능한 정보로 채움)
rows = []
# 정확도 계산: sample이 있고 ML 모델이 있다면 ML 정확도는 계산, LLM 정확도는 sample['pred']가 있으면 사용
llm_acc = None
ml_acc = None
try:
    if 'sample' in globals():
        if 'pred' in sample.columns:
            llm_acc = accuracy_score(sample['label'], sample['pred'])
        if 'model' in globals():
            ml_acc = accuracy_score(sample['label'], model.predict(sample['clean_text']))
except Exception:
    pass
# 비용 추정 (입력 토큰 기준)
PRICE_IN = 0.15 / 1_000_000
llm_cost_1e4 = None
try:
    if 'tokens' in globals() and 'sample' in globals():
        avg_tokens = tokens / len(sample)
        llm_cost_1e4 = (avg_tokens * 10000) * PRICE_IN
except Exception:
    llm_cost_1e4 = None
ml_cost = '매우 낮음 (서빙/호스팅 비용)'
rows.append({"체크": '정확도 (검증샘플)', "ML": f'{ml_acc:.3f}' if ml_acc is not None else 'N/A', "LLM": f'{llm_acc:.3f}' if llm_acc is not None else 'N/A'})
rows.append({"체크": '비용 (추정, 1만건)', "ML": ml_cost, "LLM": f'${llm_cost_1e4:.2f}' if llm_cost_1e4 is not None else '계산불가 (토큰 정보 없음)'})
rows.append({"체크": '속도', "ML": '빠름 (로컬/서빙)', "LLM": '느림 (원격 호출/토큰 처리)'} )
rows.append({"체크": '일관성', "ML": '높음 (결정적)', "LLM": '상대적으로 변동 (temperature=0으로 완화 가능)'} )
import pandas as pd
comp_df = pd.DataFrame(rows)
print()
print(comp_df.to_string(index=False))
print()
print('*대량·단순 분류는 ML, 유연·복잡 추론은 LLM*')


LLM: 와 정말 좋네요 한 번 쓰고 바로 고장나서 아주 만족합니다
  라벨: 0, 신뢰도: 0.850, 토큰: 106

LLM: 품질 최고예요~ 환불하고 싶을 만큼
  라벨: 1, 신뢰도: 0.900, 토큰: 102

LLM: 빠른 배송 감사합니다 일주일이나 걸려서요
  라벨: 0, 신뢰도: 0.850, 토큰: 102

ML 모델 학습/예측 중 오류: '[10, 20, 32, 33, 38, 42, 47, 48, 53, 57, 62, 65, 66, 72, 80, 83, 91] not found in axis'

          체크                ML                              LLM
  정확도 (검증샘플)               N/A                            1.000
비용 (추정, 1만건) 매우 낮음 (서빙/호스팅 비용)                            $0.12
          속도        빠름 (로컬/서빙)                 느림 (원격 호출/토큰 처리)
         일관성          높음 (결정적) 상대적으로 변동 (temperature=0으로 완화 가능)

*대량·단순 분류는 ML, 유연·복잡 추론은 LLM*
